In [15]:
from pathlib import Path
import duckdb as db
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
PARQUET_PATH = 'Rovshan\crimes_clean_dedup_all_years.parquet'

print(PARQUET_PATH)
print(PARQUET_PATH.exists())

c:\Users\Alejandro\Desktop\Uni Files\cbl_crime\cbl_20\Rovshan\processed\crimes_clean_dedup_all_years.parquet
False


In [17]:
#style

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

In [18]:
#quick check

db.sql(f"""
SELECT *
FROM read_parquet('{PARQUET_PATH.as_posix()}')
LIMIT 5
""").df()

IOException: IO Error: No files found that match the pattern "c:/Users/Alejandro/Desktop/Uni Files/cbl_crime/cbl_20/Rovshan/processed/crimes_clean_dedup_all_years.parquet"

In [ ]:
#time range

db.sql(f"""
SELECT MIN("Month") AS min_month, MAX("Month") AS max_month
FROM read_parquet('{PARQUET_PATH.as_posix()}')
""").df()

In [ ]:
#monthly total crime trend

monthly_total = db.sql(f"""
SELECT 
    "Month" AS month,
    COUNT(*) AS total_crimes
FROM read_parquet('{PARQUET_PATH.as_posix()}')
GROUP BY "Month"
ORDER BY "Month"
""").df()

monthly_total["month"] = pd.to_datetime(monthly_total["month"], format="%Y-%m")
monthly_total["rolling_12m"] = monthly_total["total_crimes"].rolling(12, min_periods=1).mean()

plt.figure(figsize=(14, 6))
sns.lineplot(data=monthly_total, x="month", y="total_crimes", label="monthly total")
sns.lineplot(data=monthly_total, x="month", y="rolling_12m", label="12 month rolling average")
plt.title("monthly police recorded crime over time")
plt.xlabel("month")
plt.ylabel("crime count")
plt.tight_layout()
plt.show()

In [ ]:
#monthly crime trend by category

monthly_by_type = db.sql(f"""
SELECT
    "Month" AS month,
    "Crime type" AS crime_type,
    COUNT(*) AS total_crimes
FROM read_parquet('{PARQUET_PATH.as_posix()}')
GROUP BY "Month", "Crime type"
ORDER BY "Month", total_crimes DESC
""").df()

monthly_by_type["month"] = pd.to_datetime(monthly_by_type["month"], format="%Y-%m")

top_types = (
    monthly_by_type.groupby("crime_type", as_index=False)["total_crimes"]
    .sum()
    .sort_values("total_crimes", ascending=False)
    .head(6)["crime_type"]
    .tolist()
)

monthly_top_types = monthly_by_type[monthly_by_type["crime_type"].isin(top_types)].copy()

plt.figure(figsize=(14, 7))
sns.lineplot(data=monthly_top_types, x="month", y="total_crimes", hue="crime_type")
plt.title("monthly trend for top crime categories")
plt.xlabel("month")
plt.ylabel("crime count")
plt.legend(title="crime type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
#crime category by month heatmap

heatmap_df = db.sql(f"""
SELECT
    "Month" AS month,
    "Crime type" AS crime_type,
    COUNT(*) AS total_crimes
FROM read_parquet('{PARQUET_PATH.as_posix()}')
GROUP BY "Month", "Crime type"
""").df()

heatmap_df["month"] = pd.to_datetime(heatmap_df["month"], format="%Y-%m")
heatmap_df["month_str"] = heatmap_df["month"].dt.strftime("%Y-%m")

top_types_heat = (
    heatmap_df.groupby("crime_type", as_index=False)["total_crimes"]
    .sum()
    .sort_values("total_crimes", ascending=False)
    .head(10)["crime_type"]
    .tolist()
)

heatmap_plot_df = heatmap_df[heatmap_df["crime_type"].isin(top_types_heat)].copy()
pivot_heat = heatmap_plot_df.pivot(index="crime_type", columns="month_str", values="total_crimes").fillna(0)

plt.figure(figsize=(18, 7))
sns.heatmap(pivot_heat, cmap="Oranges")
plt.title("crime category intensity by month")
plt.xlabel("month")
plt.ylabel("crime type")
plt.tight_layout()
plt.show()

In [ ]:
#force level total crime comparison

force_totals = db.sql(f"""
SELECT
    "Falls within" AS police_force,
    COUNT(*) AS total_crimes
FROM read_parquet('{PARQUET_PATH.as_posix()}')
GROUP BY "Falls within"
ORDER BY total_crimes DESC
""").df()

top_forces = force_totals.head(15).copy()

plt.figure(figsize=(12, 8))
sns.barplot(data=top_forces, y="police_force", x="total_crimes", orient="h")
plt.title("top 15 police forces by recorded crime volume")
plt.xlabel("total crimes")
plt.ylabel("police force")
plt.tight_layout()
plt.show()

In [ ]:
#force crime composition stacked bar

force_type = db.sql(f"""
SELECT
    "Falls within" AS police_force,
    "Crime type" AS crime_type,
    COUNT(*) AS total_crimes
FROM read_parquet('{PARQUET_PATH.as_posix()}')
GROUP BY "Falls within", "Crime type"
""").df()

top_force_names = force_totals.head(10)["police_force"].tolist()
top_type_names = (
    force_type.groupby("crime_type", as_index=False)["total_crimes"]
    .sum()
    .sort_values("total_crimes", ascending=False)
    .head(6)["crime_type"]
    .tolist()
)

force_type_plot = force_type[
    force_type["police_force"].isin(top_force_names) &
    force_type["crime_type"].isin(top_type_names)
].copy()

pivot_force_type = force_type_plot.pivot(index="police_force", columns="crime_type", values="total_crimes").fillna(0)
pivot_force_type = pivot_force_type.loc[top_force_names]

pivot_force_type.plot(kind="bar", stacked=True, figsize=(14, 7), colormap="tab20")
plt.title("crime composition across top 10 police forces")
plt.xlabel("police force")
plt.ylabel("total crimes")
plt.xticks(rotation=45, ha="right")
plt.legend(title="crime type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
#distribution of crime across lsoas

lsoa_counts = db.sql(f"""
SELECT
    "LSOA code" AS lsoa_code,
    COUNT(*) AS total_crimes
FROM read_parquet('{PARQUET_PATH.as_posix()}')
WHERE "LSOA code" IS NOT NULL
GROUP BY "LSOA code"
""").df()

plt.figure(figsize=(12, 6))
sns.histplot(lsoa_counts["total_crimes"], bins=60)
plt.title("distribution of total crime counts across lsoas")
plt.xlabel("total crimes per lsoa")
plt.ylabel("number of lsoas")
plt.tight_layout()
plt.show()

In [ ]:
#log distribution of crime across lsoas

lsoa_counts["log_total_crimes"] = np.log1p(lsoa_counts["total_crimes"])

plt.figure(figsize=(12, 6))
sns.histplot(lsoa_counts["log_total_crimes"], bins=60)
plt.title("log distribution of total crime counts across lsoas")
plt.xlabel("log 1 plus total crimes")
plt.ylabel("number of lsoas")
plt.tight_layout()
plt.show()

In [ ]:
#lorenz style concentration plot

lsoa_sorted = lsoa_counts.sort_values("total_crimes").reset_index(drop=True).copy()
lsoa_sorted["cum_lsoas"] = np.arange(1, len(lsoa_sorted) + 1) / len(lsoa_sorted)
lsoa_sorted["cum_crimes"] = lsoa_sorted["total_crimes"].cumsum() / lsoa_sorted["total_crimes"].sum()

plt.figure(figsize=(8, 8))
plt.plot(lsoa_sorted["cum_lsoas"], lsoa_sorted["cum_crimes"], label="crime concentration")
plt.plot([0, 1], [0, 1], linestyle="--", color="black", label="equal distribution")
plt.title("concentration of crime across lsoas")
plt.xlabel("cumulative share of lsoas")
plt.ylabel("cumulative share of crimes")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#top x percent concentration

lsoa_desc = lsoa_counts.sort_values("total_crimes", ascending=False).reset_index(drop=True).copy()
lsoa_desc["cum_crimes"] = lsoa_desc["total_crimes"].cumsum() / lsoa_desc["total_crimes"].sum()
lsoa_desc["cum_lsoas"] = (np.arange(len(lsoa_desc)) + 1) / len(lsoa_desc)

thresholds = [0.01, 0.05, 0.10, 0.20]
summary_rows = []

for t in thresholds:
    n = max(1, int(len(lsoa_desc) * t))
    share = lsoa_desc.iloc[:n]["total_crimes"].sum() / lsoa_desc["total_crimes"].sum()
    summary_rows.append({"top_share_lsoas": t, "crime_share": share})

concentration_summary = pd.DataFrame(summary_rows)
concentration_summary

In [ ]:
#crime category violin plot across forces

force_type_counts = db.sql(f"""
SELECT
    "Falls within" AS police_force,
    "Crime type" AS crime_type,
    COUNT(*) AS total_crimes
FROM read_parquet('{PARQUET_PATH.as_posix()}')
GROUP BY "Falls within", "Crime type"
""").df()

top_violin_types = (
    force_type_counts.groupby("crime_type", as_index=False)["total_crimes"]
    .sum()
    .sort_values("total_crimes", ascending=False)
    .head(8)["crime_type"]
    .tolist()
)

violin_df = force_type_counts[force_type_counts["crime_type"].isin(top_violin_types)].copy()

plt.figure(figsize=(14, 7))
sns.violinplot(data=violin_df, x="crime_type", y="total_crimes", inner="quartile", cut=0)
plt.title("distribution of force level crime counts by category")
plt.xlabel("crime type")
plt.ylabel("total crimes within force")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
#monthly crime distribution by category using violin

monthly_type_counts = db.sql(f"""
SELECT
    "Month" AS month,
    "Crime type" AS crime_type,
    COUNT(*) AS total_crimes
FROM read_parquet('{PARQUET_PATH.as_posix()}')
GROUP BY "Month", "Crime type"
""").df()

top_monthly_violin_types = (
    monthly_type_counts.groupby("crime_type", as_index=False)["total_crimes"]
    .sum()
    .sort_values("total_crimes", ascending=False)
    .head(8)["crime_type"]
    .tolist()
)

monthly_violin_df = monthly_type_counts[
    monthly_type_counts["crime_type"].isin(top_monthly_violin_types)
].copy()

plt.figure(figsize=(14, 7))
sns.violinplot(data=monthly_violin_df, x="crime_type", y="total_crimes", inner="quartile", cut=0)
plt.title("distribution of monthly crime counts by category")
plt.xlabel("crime type")
plt.ylabel("monthly total crimes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
#top forces over time

monthly_force = db.sql(f"""
SELECT
    "Month" AS month,
    "Falls within" AS police_force,
    COUNT(*) AS total_crimes
FROM read_parquet('{PARQUET_PATH.as_posix()}')
GROUP BY "Month", "Falls within"
""").df()

monthly_force["month"] = pd.to_datetime(monthly_force["month"], format="%Y-%m")

top_force_names_time = (
    monthly_force.groupby("police_force", as_index=False)["total_crimes"]
    .sum()
    .sort_values("total_crimes", ascending=False)
    .head(6)["police_force"]
    .tolist()
)

monthly_force_plot = monthly_force[monthly_force["police_force"].isin(top_force_names_time)].copy()

plt.figure(figsize=(14, 7))
sns.lineplot(data=monthly_force_plot, x="month", y="total_crimes", hue="police_force")
plt.title("monthly crime trends for top police forces")
plt.xlabel("month")
plt.ylabel("total crimes")
plt.legend(title="police force", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
#force by crime type heatmap

force_type_heat = db.sql(f"""
SELECT
    "Falls within" AS police_force,
    "Crime type" AS crime_type,
    COUNT(*) AS total_crimes
FROM read_parquet('{PARQUET_PATH.as_posix()}')
GROUP BY "Falls within", "Crime type"
""").df()

top_force_heat = (
    force_type_heat.groupby("police_force", as_index=False)["total_crimes"]
    .sum()
    .sort_values("total_crimes", ascending=False)
    .head(15)["police_force"]
    .tolist()
)

top_type_heat = (
    force_type_heat.groupby("crime_type", as_index=False)["total_crimes"]
    .sum()
    .sort_values("total_crimes", ascending=False)
    .head(10)["crime_type"]
    .tolist()
)

force_type_heat_plot = force_type_heat[
    force_type_heat["police_force"].isin(top_force_heat) &
    force_type_heat["crime_type"].isin(top_type_heat)
].copy()

pivot_force_heat = force_type_heat_plot.pivot(index="police_force", columns="crime_type", values="total_crimes").fillna(0)

plt.figure(figsize=(14, 8))
sns.heatmap(pivot_force_heat, cmap="Oranges")
plt.title("force by crime type intensity")
plt.xlabel("crime type")
plt.ylabel("police force")
plt.tight_layout()
plt.show()